# Tenacious-Bench v0.1 — ORPO LoRA Training (Unsloth)

**Path B**: ORPO preference optimization on Qwen3.5-0.8B via Unsloth.

- ORPO combines SFT and preference loss in one pass — no reference model needed
- More data-efficient than SimPO for small preference pair datasets (~158 pairs)
- Unsloth uses ~50% less VRAM and runs ~1.5x faster than vanilla TRL
- 16-bit LoRA (fp16 on T4, bf16 on A100/4090) — per Unsloth Qwen3.5 guide
- Backbone: `unsloth/Qwen3.5-0.8B`

**Target metrics**:
- Delta A (trained vs Week 10 baseline on held-out): > 0 pts
- Delta B (trained vs prompt-engineered base on held-out): > 0 pts

In [ ]:
# Step 1 — Install Unsloth (handles transformers/trl/peft versions automatically)
!pip install -q unsloth
!pip install -q --upgrade --no-deps unsloth_zoo

# Restart runtime after install so numpy loads cleanly
import os
os.kill(os.getpid(), 9)

In [ ]:
# Step 2 — Verify GPU (run after runtime restarts)
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Step 3 — Upload the 4 required files (one dialog per file)
from google.colab import files
import os, json

print("1/4 — Navigate to: week11/training_data/preference_pairs.jsonl")
files.upload()

print("2/4 — Navigate to: week11/tenacious_bench_v0.1/dev/dev.jsonl")
files.upload()

print("3/4 — Navigate to: week11/tenacious_bench_v0.1/held_out/held_out.jsonl")
files.upload()

print("4/4 — Navigate to: week11/scoring_evaluator.py")
files.upload()

PAIRS_FILE    = 'preference_pairs.jsonl'
DEV_FILE      = 'dev.jsonl'
HELD_OUT_FILE = 'held_out.jsonl'

for f in [PAIRS_FILE, DEV_FILE, HELD_OUT_FILE, 'scoring_evaluator.py']:
    print(f"  {'OK' if os.path.exists(f) else 'MISSING'} — {f}")

In [ ]:
# Step 4 — Load and inspect preference pairs
from datasets import Dataset
from collections import Counter

pairs = [json.loads(l) for l in open(PAIRS_FILE)]
print(f"Loaded {len(pairs)} preference pairs")

dims = Counter(p['dimension'] for p in pairs)
for d, n in sorted(dims.items()):
    print(f"  {d:40s} {n}")

dataset = Dataset.from_list([
    {'prompt': p['prompt'], 'chosen': p['chosen'], 'rejected': p['rejected']}
    for p in pairs
])
print(f"\nDataset: {dataset}")

In [ ]:
# Step 5 — Load model with Unsloth (16-bit LoRA, as per Unsloth Qwen3.5 guide)
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024
MODEL_ID = "unsloth/Qwen3.5-0.8B"   # challenge-recommended smallest backbone

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,  # 16-bit LoRA; precision follows GPU native support (fp16 on T4)
    dtype=None,          # auto-detect: fp16 on T4, bf16 on A100/4090
)

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

In [ ]:
# Step 6 — Attach LoRA adapter via Unsloth
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient checkpointing
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# Step 6b — Delta B baseline: untrained model + explicit policy prompt
# MUST run BEFORE training (Step 8). Measures whether training beats prompting alone.
# The LoRA adapter is attached but weights are still at initialisation (near-zero effect).
import re, torch, json, sys

POLICY_PROMPT = """You are the Tenacious sales agent. Follow these rules exactly:
1. If reply_intent=POSITIVE and confidence>=0.65: action=send_followup, autonomous=True.
2. If reply_intent=POSITIVE and confidence<0.65: action=send_clarifying_question, autonomous=True.
3. If the prospect asks about pricing or wants to sign legal docs: action=route_to_human, escalation_reason=pricing|legal.
4. If prospect says unsubscribe/GDPR: action=no_action.
5. Never claim bench capacity you do not have. If stack is unavailable: admit it or route_to_human.
6. Never use aggressive language: forbidden words include "scaling aggressively", "aggressive hiring".
7. Always ground claims in the provided signal. Hedge when signal is weak or single-source.

Output format (fill in each line):
ACTION: <action>
AUTONOMOUS: <True|False>
SUBJECT: <subject line, max 60 chars>
EMAIL:
<email body, max 120 words, end with gettenacious.com signature>
ESCALATION_REASON: <reason or None>
"""

def build_prompt_with_policy(task):
    inp = task.get('input', {})
    ctx = inp.get('prospect_context', {})
    bench = inp.get('bench_state', {})
    lines = [
        POLICY_PROMPT,
        "--- Task ---",
        f"Company: {ctx.get('company_name','')}",
        f"Funding: {ctx.get('funding_amount','')} {ctx.get('funding_round','')}",
        f"Employees: {ctx.get('employee_count','')}",
        f"Eng roles open: {ctx.get('engineering_roles_open','')}",
        f"AI maturity: {ctx.get('ai_maturity_score','')}",
        f"Signal sources: {', '.join(ctx.get('signal_sources',[]))}",
        f"Bench state: {json.dumps(bench)}",
    ]
    reply = inp.get('prospect_reply')
    if reply:
        lines += [f"Reply: {reply}", f"Intent: {inp.get('reply_intent','')}", f"Confidence: {inp.get('confidence','')}"]
    return "\n".join(lines)

def parse_output(text):
    # Strip Qwen3.5 thinking blocks before parsing — thinking mode generates
    # <think>...</think> before the actual answer; match against the answer only.
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    action_m = re.search(r'ACTION:\s*(\S+)', text)
    auto_m   = re.search(r'AUTONOMOUS:\s*(True|False)', text)
    email_m  = re.search(r'EMAIL:\n(.+?)(?=\nESCALATION_REASON:|$)', text, re.DOTALL)
    subj_m   = re.search(r'SUBJECT:\s*(.+)', text)
    esc_m    = re.search(r'ESCALATION_REASON:\s*(.+)', text)
    return {
        'action':            action_m.group(1) if action_m else 'unknown',
        'autonomous':        auto_m.group(1) == 'True' if auto_m else False,
        'email_body':        email_m.group(1).strip() if email_m else None,
        'subject_line':      subj_m.group(1).strip() if subj_m else None,
        'capacity_claim':    None,
        'escalation_reason': esc_m.group(1).strip() if esc_m and esc_m.group(1).strip() != 'None' else None,
        'raw_generation':    text,
    }

def generate_with_base(task):
    prompt = build_prompt_with_policy(task)
    # text= keyword required — Unsloth patches tokenizer with VL signature (images, text, videos)
    # positional arg would land in 'images' and crash; explicit text= bypasses VL routing
    inputs = tokenizer(text=prompt, return_tensors='pt', truncation=True, max_length=900).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return parse_output(text)

# Load held_out now so Delta B baseline can run before training
sys.path.insert(0, '.')
import scoring_evaluator as se
held_tasks_for_deltab = [json.loads(l) for l in open(HELD_OUT_FILE)]

print("Running Delta B baseline (untrained model + policy prompt) on held-out...")
deltab_results = []
for t in held_tasks_for_deltab:
    gen = generate_with_base(t)
    score_result = se.score_task(t, gen)
    deltab_results.append({
        'task_id': t.get('task_id'),
        'score': score_result.get('total_score', 0),
        'passed': score_result.get('passed', False),
    })

deltab_held_avg = sum(r['score'] for r in deltab_results) / len(deltab_results)
deltab_held_scores = [r['score'] for r in deltab_results]
print(f"Delta B baseline (untrained model + policy prompt): held-out avg = {deltab_held_avg:.3f}")
print("Training will now proceed in Steps 7-9. Compare this score to Step 11.")

In [ ]:
# Step 6c — Verify think-block stripping works before training
# parse_output defined in Step 6b; this cell confirms it correctly removes <think>...</think>
import re

_sample = (
    "<think>The prospect has high AI score and positive intent. "
    "I should send a followup autonomously.</think>\n"
    "ACTION: send_followup\n"
    "AUTONOMOUS: True\n"
    "SUBJECT: Following up on your Series B interest\n"
    "EMAIL:\nHi — wanted to follow up on our earlier conversation."
)
_stripped = re.sub(r'<think>.*?</think>', '', _sample, flags=re.DOTALL).strip()
assert '<think>' not in _stripped, "FAIL: think block not removed"
assert 'ACTION:' in _stripped,     "FAIL: action missing after stripping"
print("Think-block stripping: PASS")
print(f"  Input length : {len(_sample)} chars")
print(f"  Output length: {len(_stripped)} chars")
print(f"  First 80 chars of stripped output: {_stripped[:80]}")

In [ ]:
# Step 7 — Configure ORPO trainer
# ORPO (Hong et al., EMNLP 2024): combines SFT + odds-ratio preference loss in one pass.
# No reference model needed — more data-efficient than SimPO for small datasets.
# beta=0.2: stronger preference signal than default 0.1, helps on categorical policy failures
# lr=2e-5: overcomes base model bias in 750 steps at 0.8B scale
from trl import ORPOConfig, ORPOTrainer
import torch

training_args = ORPOConfig(
    output_dir='./checkpoints',
    beta=0.2,                        # odds-ratio weight; 0.2 strengthens preference loss for categorical failures
    learning_rate=2e-5,              # 2e-5 overcomes base model bias in 750 steps at 0.8B scale
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16
    max_steps=750,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    fp16=not torch.cuda.is_bf16_supported(),   # fp16 on T4, bf16 on A100/4090
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    save_steps=250,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=768,
    remove_unused_columns=False,
    report_to='none',
    optim='adamw_8bit',              # Unsloth 8-bit optimizer
)

trainer = ORPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("ORPO Trainer configured. Starting training...")

In [ ]:
# Step 8 — Train (~15 min on T4)
train_result = trainer.train()
print(train_result)

In [ ]:
# Step 9 — Save LoRA adapter
ADAPTER_DIR = './tenacious-orpo-lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

## Ablations — Delta A and Delta B

In [ ]:
# Step 10 — Baseline: score dev + held-out using the embedded candidate outputs
import sys
sys.path.insert(0, '.')
import scoring_evaluator as se

dev_tasks  = [json.loads(l) for l in open(DEV_FILE)]
held_tasks = [json.loads(l) for l in open(HELD_OUT_FILE)]
print(f"Dev: {len(dev_tasks)} tasks   Held-out: {len(held_tasks)} tasks")

dev_dc   = [t for t in dev_tasks  if t.get('dimension') == 'dual_control_decision']
dev_sg   = [t for t in dev_tasks  if t.get('dimension') == 'signal_grounding']
held_dc  = [t for t in held_tasks if t.get('dimension') == 'dual_control_decision']
held_sg  = [t for t in held_tasks if t.get('dimension') == 'signal_grounding']

print(f"  Dev  — dual_control: {len(dev_dc)}  signal_grounding: {len(dev_sg)}")
print(f"  Held — dual_control: {len(held_dc)}  signal_grounding: {len(held_sg)}")

def avg_score(tasks):
    # score_task(task, output) — two required args
    scores = [se.score_task(t, t['candidate_output']).get('total_score', 0) for t in tasks]
    return sum(scores) / len(scores) if scores else 0.0

baseline_dev_dc  = avg_score(dev_dc)
baseline_dev_sg  = avg_score(dev_sg)
baseline_held_all = avg_score(held_tasks)
baseline_held_dc  = avg_score(held_dc)
baseline_held_sg  = avg_score(held_sg)
# Per-task scores for bootstrap test later
baseline_held_scores = [se.score_task(t, t['candidate_output']).get('total_score', 0) for t in held_tasks]

print(f"\nBaseline (dev)  — dual_control: {baseline_dev_dc:.3f}  signal_grounding: {baseline_dev_sg:.3f}")
print(f"Baseline (held) — overall: {baseline_held_all:.3f}  dual_control: {baseline_held_dc:.3f}  signal_grounding: {baseline_held_sg:.3f}")

In [ ]:
# Step 11 — Post-training: generate and score dev + held-out with the trained adapter
import time
FastLanguageModel.for_inference(model)  # Unsloth 2x inference mode

def build_prompt(task):
    """Exact match to format_preference_pairs.py build_prompt — training and eval must use identical format."""
    inp = task.get('input', {})
    ctx = inp.get('prospect_context', {})
    bench = inp.get('bench_state', {})
    thread = inp.get('prior_thread', [])
    reply = inp.get('prospect_reply')
    intent = inp.get('reply_intent')
    confidence = inp.get('confidence')

    lines = ["<|system|>"]
    lines.append("You are the Tenacious Intelligence Corporation sales agent.")
    lines.append(
        "Tenacious provides engineering staff augmentation. "
        "You classify prospects into ICP segments, generate grounded outreach emails, "
        "and decide when to act autonomously vs. escalate to a human."
    )
    lines.append("")
    lines.append("Bench state (available engineers):")
    for stack, info in bench.items():
        lines.append(f"  {stack}: {info.get('available_engineers', 0)} available")
    lines.append("")
    lines.append("Policies:")
    lines.append("  - Act autonomously only if reply_intent is POSITIVE AND confidence >= 0.65")
    lines.append("  - Escalate for: pricing questions, legal/NDA requests, GDPR questions, unsubscribe")
    lines.append("  - Never commit bench capacity beyond what is available")
    lines.append("  - Outreach emails: max 120 words, direct, grounded, honest, professional, non-condescending")
    lines.append("<|end_system|>")
    lines.append("")
    lines.append("<|context|>")

    company = ctx.get('company_name', 'Unknown')
    funding = ctx.get('funding_amount')
    round_ = ctx.get('funding_round', '')
    employees = ctx.get('employee_count')
    roles_open = ctx.get('engineering_roles_open')
    layoffs = ctx.get('layoff_events', [])
    ai_score = ctx.get('ai_maturity_score')
    sources = ctx.get('signal_sources', [])

    lines.append(f"Company: {company}")
    if funding:
        lines.append(f"Funding: ${funding:,} {round_}".strip())
    if employees:
        lines.append(f"Employee count: {employees}")
    if roles_open is not None:
        lines.append(f"Engineering roles open: {roles_open}")
    if layoffs:
        for le in layoffs:
            date_str = le.get('date') or 'undated'
            pct = le.get('pct_cut', '')
            lines.append(f"Layoff event: {pct}% cut, date={date_str}")
    if ai_score is not None:
        lines.append(f"AI maturity score: {ai_score}/4")
    if sources:
        lines.append(f"Signal sources: {', '.join(sources)}")

    if thread:
        lines.append("")
        lines.append("Prior conversation thread:")
        for turn in thread:
            role = turn.get('role', 'unknown')
            content = turn.get('content', '')
            lines.append(f"  [{role}]: {content}")

    if reply:
        lines.append("")
        lines.append(f"Prospect reply: {reply}")
        if intent:
            lines.append(f"Reply intent: {intent}")
        if confidence is not None:
            lines.append(f"Confidence: {confidence}")

    lines.append("<|end_context|>")
    lines.append("")
    lines.append("<|task|>Decide the correct action and generate the appropriate output.<|end_task|>")

    return "\n".join(lines)

def generate_trained(task):
    prompt = build_prompt(task)
    t0 = time.time()
    # text= keyword required — Unsloth patches tokenizer with VL signature (images, text, videos)
    # positional arg would land in 'images' and crash; explicit text= bypasses VL routing
    inputs = tokenizer(text=prompt, return_tensors='pt', truncation=True, max_length=900).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    latency_ms = (time.time() - t0) * 1000
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed = parse_output(text)   # parse_output strips <think>...</think> blocks
    parsed['latency_ms'] = round(latency_ms, 1)
    return parsed

def score_and_trace(tasks, split_name):
    traces = []
    t_start = time.time()
    for t in tasks:
        gen = generate_trained(t)
        result = se.score_task(t, gen)
        traces.append({
            'task_id':          t.get('task_id'),
            'split':            split_name,
            'dimension':        t.get('dimension'),
            'difficulty':       t.get('difficulty'),
            'generated_output': {k: v for k, v in gen.items() if k != 'raw_generation'},
            'raw_generation':   gen.get('raw_generation', ''),
            'score':            result.get('total_score', 0),
            'passed':           result.get('passed', False),
            'latency_ms':       gen.get('latency_ms', 0),
        })
    total_s = time.time() - t_start
    avg = sum(r['score'] for r in traces) / len(traces) if traces else 0
    print(f"  {split_name}: {len(traces)} tasks, avg={avg:.3f}, {total_s:.0f}s total")
    return traces

print("Evaluating trained adapter on dev...")
dev_traces = score_and_trace(dev_tasks, 'dev')

print("Evaluating trained adapter on held-out (official Delta A)...")
held_traces = score_and_trace(held_tasks, 'held_out')

# Summary scores
trained_dev_all  = sum(r['score'] for r in dev_traces)  / len(dev_traces)
trained_held_all = sum(r['score'] for r in held_traces) / len(held_traces)
trained_held_scores = [r['score'] for r in held_traces]

trained_held_dc = [r['score'] for r in held_traces if r['dimension'] == 'dual_control_decision']
trained_held_sg = [r['score'] for r in held_traces if r['dimension'] == 'signal_grounding']
trained_held_dc_avg = sum(trained_held_dc)/len(trained_held_dc) if trained_held_dc else 0
trained_held_sg_avg = sum(trained_held_sg)/len(trained_held_sg) if trained_held_sg else 0

delta_a = (trained_held_all - baseline_held_all) * 100
delta_b = (trained_held_all - deltab_held_avg) * 100

print(f"\n=== Preliminary Results ===")
print(f"  Held-out baseline:        {baseline_held_all:.3f}")
print(f"  Held-out prompt-eng (B):  {deltab_held_avg:.3f}")
print(f"  Held-out trained:         {trained_held_all:.3f}")
print(f"  Delta A (vs baseline):   {delta_a:+.1f} pts")
print(f"  Delta B (vs prompt-eng): {delta_b:+.1f} pts")

In [ ]:
# Step 12 — Statistical significance + save all artifacts
import random, zipfile

def paired_bootstrap(baseline_scores, trained_scores, n_bootstrap=2000, seed=42):
    """One-sided paired bootstrap: P(trained > baseline). Returns p-value."""
    random.seed(seed)
    n = len(baseline_scores)
    diffs = [t - b for t, b in zip(trained_scores, baseline_scores)]
    observed_mean = sum(diffs) / n
    centred = [d - observed_mean for d in diffs]
    count = sum(
        sum(random.choice(centred) for _ in range(n)) / n >= observed_mean
        for _ in range(n_bootstrap)
    )
    p = count / n_bootstrap
    boot_means = sorted(sum(random.choices(diffs, k=n)) / n for _ in range(n_bootstrap))
    ci_lo = boot_means[int(0.025 * n_bootstrap)]
    ci_hi = boot_means[int(0.975 * n_bootstrap)]
    return {'observed_mean_diff': round(observed_mean, 4),
            'p_value': round(p, 4),
            'significant_p05': p < 0.05,
            'ci_95': [round(ci_lo, 4), round(ci_hi, 4)]}

stat_a = paired_bootstrap(baseline_held_scores, trained_held_scores)
stat_b = paired_bootstrap(deltab_held_scores,   trained_held_scores)

avg_latency_ms = sum(r['latency_ms'] for r in held_traces) / len(held_traces)

print("=== Final Ablation Results ===")
print(f"\nDelta A (trained vs Week-10 baseline, held-out)")
print(f"  Baseline:  {baseline_held_all:.4f}")
print(f"  Trained:   {trained_held_all:.4f}")
print(f"  Lift:      {delta_a:+.2f} pts")
print(f"  95% CI:    [{stat_a['ci_95'][0]*100:+.2f}, {stat_a['ci_95'][1]*100:+.2f}] pts")
print(f"  p-value:   {stat_a['p_value']}  ({'SIGNIFICANT' if stat_a['significant_p05'] else 'not significant'})")

print(f"\nDelta B (trained vs prompt-engineered base, held-out)")
print(f"  Prompt-eng: {deltab_held_avg:.4f}")
print(f"  Trained:    {trained_held_all:.4f}")
print(f"  Lift:       {delta_b:+.2f} pts")
print(f"  p-value:    {stat_b['p_value']}  ({'SIGNIFICANT' if stat_b['significant_p05'] else 'not significant'})")

print(f"\nCost-Pareto")
print(f"  Avg latency per task (with adapter): {avg_latency_ms:.0f} ms")
print(f"  Baseline latency: ~0 ms (no generation)")

# --- Save ablation_results.json ---
ablation_results = {
    'model': MODEL_ID,
    'adapter': 'tenacious-orpo-lora',
    'lora_config': {'r': 32, 'lora_alpha': 64, 'lora_dropout': 0.05,
                    'target_modules': ['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj']},
    'orpo_config': {'beta': 0.2, 'lr': 2e-5, 'steps': 750,
                    'batch_size': 2, 'gradient_accumulation': 8, 'warmup_steps': 50},
    'training_pairs': len(pairs),
    'seed': 42,
    'baseline': {
        'held_out_avg': round(baseline_held_all, 4),
        'held_out_dc':  round(baseline_held_dc, 4),
        'held_out_sg':  round(baseline_held_sg, 4),
    },
    'prompt_engineered': {
        'held_out_avg': round(deltab_held_avg, 4),
    },
    'post_training': {
        'held_out_avg': round(trained_held_all, 4),
        'held_out_dc':  round(trained_held_dc_avg, 4),
        'held_out_sg':  round(trained_held_sg_avg, 4),
    },
    'delta_a': {
        'lift_pts': round(delta_a, 2),
        'ci_95_pts': [round(stat_a['ci_95'][0]*100, 2), round(stat_a['ci_95'][1]*100, 2)],
        'p_value': stat_a['p_value'],
        'significant': stat_a['significant_p05'],
    },
    'delta_b': {
        'lift_pts': round(delta_b, 2),
        'p_value': stat_b['p_value'],
        'significant': stat_b['significant_p05'],
    },
    'cost_pareto': {
        'avg_latency_ms_with_adapter': round(avg_latency_ms, 1),
        'avg_latency_ms_baseline': 0,
    },
    'train_metrics': train_result.metrics if hasattr(train_result, 'metrics') else {},
}

with open('ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)
print("\nablation_results.json written")

# --- Save held_out_traces.jsonl ---
with open('held_out_traces.jsonl', 'w') as f:
    for row in held_traces:
        f.write(json.dumps(row) + '\n')
print(f"held_out_traces.jsonl written ({len(held_traces)} rows)")

# --- Zip both into one file and download once (avoids browser popup blocker) ---
with zipfile.ZipFile('orpo_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('ablation_results.json')
    zf.write('held_out_traces.jsonl')
print("orpo_results.zip created")

files.download('orpo_results.zip')
print("\nDownload triggered: orpo_results.zip")
print("Unzip it and place both files in: week11/ablations/")